# 07 — ONNX Export & Edge Benchmark

**DRISHTI** — Export best YOLOv8-seg to ONNX, quantize INT8, benchmark for edge deployment

Target: **Tier 0** — <15MB model, <200ms/image on CPU.

**Run on:** Colab/Kaggle (GPU for export, CPU for benchmarks).

In [ ]:
!pip install -q ultralytics onnx onnxruntime onnxsim numpy matplotlib

In [ ]:
import numpy as np
import time
import json
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

MODEL_PATH = 'drishti_training/runs/yolov8s_seg_baseline/weights/best.pt'
OUTPUT_DIR = Path('exported_models')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Model: {MODEL_PATH}')
print(f'Output: {OUTPUT_DIR}')

## 1. ONNX Export

In [ ]:
model = YOLO(MODEL_PATH)

# Export to ONNX
export_path = model.export(
    format='onnx',
    imgsz=640,
    opset=12,
    simplify=True,
    half=False,
    dynamic=False,
)

onnx_path = Path(export_path)
print(f'\nONNX exported: {onnx_path}')
print(f'Size: {onnx_path.stat().st_size / 1e6:.2f} MB')

## 2. Validate ONNX vs PyTorch

In [ ]:
import onnx
import onnxruntime as ort

# Validate model structure
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
print('✓ ONNX model structure is valid')

# Print model info
print(f'  IR version: {onnx_model.ir_version}')
print(f'  Opset: {onnx_model.opset_import[0].version}')
print(f'  Inputs:')
for inp in onnx_model.graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f'    {inp.name}: {shape}')
print(f'  Outputs: {len(onnx_model.graph.output)}')

# Compare outputs
test_imgs = sorted(Path('drishti_training/data/splits/test/images').glob('*.png'))[:5]

pt_model = YOLO(MODEL_PATH)
onnx_model_yolo = YOLO(str(onnx_path))

print('\nOutput comparison (PT vs ONNX):')
for img_path in test_imgs:
    pt_res = pt_model.predict(str(img_path), conf=0.25, verbose=False)
    ox_res = onnx_model_yolo.predict(str(img_path), conf=0.25, verbose=False)
    
    pt_n = len(pt_res[0].boxes) if pt_res[0].boxes is not None else 0
    ox_n = len(ox_res[0].boxes) if ox_res[0].boxes is not None else 0
    match = '✓' if abs(pt_n - ox_n) <= 2 else '✗'
    print(f'  {img_path.name}: PT={pt_n}, ONNX={ox_n} [{match}]')

## 3. INT8 Quantization (Optional)

In [ ]:
try:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    
    int8_path = OUTPUT_DIR / f'{onnx_path.stem}_int8.onnx'
    
    quantize_dynamic(
        model_input=str(onnx_path),
        model_output=str(int8_path),
        weight_type=QuantType.QUInt8,
    )
    
    orig_size = onnx_path.stat().st_size / 1e6
    int8_size = int8_path.stat().st_size / 1e6
    print(f'INT8 quantized: {int8_path}')
    print(f'Size: {int8_size:.2f} MB ({int8_size/orig_size*100:.0f}% of original {orig_size:.2f} MB)')
    
except Exception as e:
    print(f'Quantization skipped: {e}')
    int8_path = None

## 4. Latency Benchmark

In [ ]:
def benchmark(model_path, name, n_warmup=10, n_runs=50, imgsz=640):
    session = ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name
    input_shape = session.get_inputs()[0].shape
    
    # Create dummy input
    channels = input_shape[1] if len(input_shape) > 1 else 3
    dummy = np.random.randn(1, channels, imgsz, imgsz).astype(np.float32)
    
    # Warmup
    for _ in range(n_warmup):
        session.run(None, {input_name: dummy})
    
    # Benchmark
    latencies = []
    for _ in range(n_runs):
        start = time.perf_counter()
        session.run(None, {input_name: dummy})
        latencies.append((time.perf_counter() - start) * 1000)
    
    latencies = np.array(latencies)
    size_mb = Path(model_path).stat().st_size / 1e6
    
    result = {
        'name': name,
        'size_mb': size_mb,
        'mean_ms': latencies.mean(),
        'median_ms': np.median(latencies),
        'p95_ms': np.percentile(latencies, 95),
        'fps': 1000 / latencies.mean(),
        'latencies': latencies,
    }
    
    print(f'\n{name}:')
    print(f'  Size:   {size_mb:.2f} MB')
    print(f'  Mean:   {result["mean_ms"]:.1f} ms')
    print(f'  Median: {result["median_ms"]:.1f} ms')
    print(f'  P95:    {result["p95_ms"]:.1f} ms')
    print(f'  FPS:    {result["fps"]:.1f}')
    
    # Tier 0 check
    tier0 = size_mb < 15 and result['mean_ms'] < 200
    print(f'  Tier 0 (<15MB, <200ms): {"PASS ✓" if tier0 else "FAIL ✗"}')
    
    return result

results = []
results.append(benchmark(onnx_path, 'FP32 ONNX'))
if int8_path and int8_path.exists():
    results.append(benchmark(int8_path, 'INT8 ONNX'))

In [ ]:
# ---- Visualise benchmarks ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

names = [r['name'] for r in results]
colors = ['steelblue', 'coral', 'green'][:len(results)]

# Size comparison
sizes = [r['size_mb'] for r in results]
bars = axes[0].bar(names, sizes, color=colors, alpha=0.8)
axes[0].axhline(y=15, color='red', linestyle='--', label='Tier 0 limit (15MB)')
axes[0].set_ylabel('Size (MB)')
axes[0].set_title('Model Size')
axes[0].legend()
for bar, val in zip(bars, sizes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}MB', ha='center', fontsize=10)

# Latency comparison
means = [r['mean_ms'] for r in results]
bars = axes[1].bar(names, means, color=colors, alpha=0.8)
axes[1].axhline(y=200, color='red', linestyle='--', label='Tier 0 limit (200ms)')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Mean Inference Latency (CPU)')
axes[1].legend()
for bar, val in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{val:.0f}ms', ha='center', fontsize=10)

# Latency distribution
for r, c in zip(results, colors):
    axes[2].hist(r['latencies'], bins=20, alpha=0.5, color=c, label=r['name'])
axes[2].set_xlabel('Latency (ms)')
axes[2].set_ylabel('Count')
axes[2].set_title('Latency Distribution')
axes[2].legend()

plt.suptitle('ONNX Export — Edge Deployment Benchmarks', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Edge-Readiness Report

In [ ]:
print('=' * 60)
print('DRISHTI — Edge Deployment Readiness Report')
print('=' * 60)
print()

for r in results:
    tier0_size = r['size_mb'] < 15
    tier0_speed = r['mean_ms'] < 200
    tier0 = tier0_size and tier0_speed
    
    print(f'{r["name"]:20s}')
    print(f'  Size:     {r["size_mb"]:>8.2f} MB  {"✓" if tier0_size else "✗"} (<15MB)')
    print(f'  Latency:  {r["mean_ms"]:>8.1f} ms  {"✓" if tier0_speed else "✗"} (<200ms)')
    print(f'  FPS:      {r["fps"]:>8.1f}')
    print(f'  Tier 0:   {"PASS ✓" if tier0 else "FAIL ✗"}')
    print()

print('Deployment targets:')
print('  Tier 0: Laptop/server CPU    → ONNX FP32 or INT8')
print('  Tier 1: NVIDIA Jetson        → ONNX + TensorRT (onnxruntime-gpu)')
print()
print('Files for deployment:')
print(f'  ONNX model:   {onnx_path}')
if int8_path:
    print(f'  INT8 model:   {int8_path}')
print(f'  Calibrator:   ml/models/exported/calibrator.pkl')
print(f'  Config:       ml/configs/drishti.yaml')
print()
print('These feed into edge/onnx_runtime_server.py and backend/detections/tasks.py')

## Summary

| Metric | FP32 | INT8 |
|--------|------|------|
| Size | (see above) | (see above) |
| Latency | (see above) | (see above) |
| Tier 0 | (see above) | (see above) |

The full pipeline is now complete:
1. ✅ Dataset explored & prepared
2. ✅ YOLOv8s-seg fine-tuned from COCO weights
3. ✅ Synthetic rare-class data generated
4. ✅ Confidence calibrated (Platt scaling)
5. ✅ Errors analysed (FP/FN galleries)
6. ✅ Shadow geometry filter implemented
7. ✅ ONNX exported + benchmarked for edge